# M1 实验队列 · Kaggle Runner

**用法**:Kaggle → New Notebook → File → Import Notebook 粘贴本文件(或从 GitHub `AwareLiquid/M1` 的 `kaggle/kaggle_runner.ipynb` 导入)→ 右侧 Settings 选 **GPU** → Run All。

- 每个实验独立,跑完一个就把结果(`reasoning_depth.jsonl` + log)拷到 `/kaggle/working/`,**12h 会话被杀也不丢已完成的**。
- 会话结束后在 Output 面板下载 `m1_results.zip`,发回给 CC 入库。
- 断点续跑:把下面 `START` 改成上次完成的序号+1,重新 Run All。

**当前队列(2026-08-06)— parity k=32 长预算裁决 + J-Space J1 sweep + 125M selective_decay**:
- parity k=32 上次内核在 selective 臂 step 8600/12000 处被取消(`CANCEL_ACKNOWLEDGED`),未出结果;本地 6000 步 d32 全 chance(含 transformer 0.557)= 预算墙。
- **P100(sm_60)解锁**:默认 Kaggle torch(≥2.2, cu121/cu124)移除了 sm_60 → 需安装 cu118 构建(`torch 2.1.2+cu118`, arch 含 sm_60)才能用 GPU。runner 首个 cell 自动处理。
- J1 是 workspace 驻留深度探针。纯 LNN 探针(`--attention_layers` 空)是有效协议(hybrid 被 attention 兜底,已证 null)。

In [ ]:
import subprocess, os, shutil, time, glob, sys

# ── 环境:克隆仓库 + P100(sm_60)解锁 ──
# 默认 Kaggle torch(≥2.2 cu121/cu124)移除了 sm_60 → P100 无 CUDA。
# cu118 构建保留 sm_60(sm_37..sm_90)→ 装 cu118 torch 即可在 P100 上跑 GPU。
# ⚠ 注意: Kaggle 内核是 Python 3.12 → 必须用 torch==2.3.1+cu118(有 cp312 wheel);
#   2.1.2+cu118 无 cp312 wheel,pip 会安装失败(2026-08-06 实测)。
if not os.path.exists('/kaggle/working/M1'):
    subprocess.run(['git','clone','--depth=1','https://github.com/AwareLiquid/M1.git','/kaggle/working/M1'], check=True)
os.chdir('/kaggle/working/M1')
subprocess.run(['git','pull','--ff-only'], check=False)

# 用 nvidia-smi 探测 GPU(不 import torch,避免锁住旧版本)
gpu_name, cap = '', (0, 0)
try:
    out = subprocess.run(['nvidia-smi','--query-gpu=name,compute_cap','--format=csv,noheader'],
                         capture_output=True, text=True, check=True)
    line = out.stdout.strip().splitlines()[0] if out.stdout.strip() else ''
    if line:
        parts = [p.strip() for p in line.split(',')]
        gpu_name = parts[0]
        cap = tuple(int(x) for x in parts[1].split('.')) if len(parts) > 1 else (0, 0)
except Exception as e:
    print('nvidia-smi failed:', e, flush=True)
print('gpu:', gpu_name, 'capability', cap, flush=True)

# 若 P100 且当前 torch 无 sm_60 → 先装 cu118 torch 再 import
if cap[0] == 6:
    try:
        import torch
        if 'sm_60' not in (torch.cuda.get_arch_list() or []):
            raise RuntimeError('no sm_60')
    except Exception:
        print('P100 检测 + 当前 torch 无 sm_60 → 安装 cu118 torch(保留 Pascal 支持)', flush=True)
        try:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                            'torch==2.3.1+cu118', 'torchvision==0.18.1+cu118',
                            '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
            print('cu118 torch 安装成功', flush=True)
        except Exception as e:
            print(f'⚠ cu118 安装失败: {e}', flush=True)
            print('  回退 CPU 模式: 队列 A/B/C(200K 探针)仍可跑,125M(D) 跳过', flush=True)

import torch
ok = torch.cuda.is_available()
name = torch.cuda.get_device_name(0) if ok else ''
cap2 = torch.cuda.get_device_capability(0) if ok else (0, 0)
arch = torch.cuda.get_arch_list() if ok else []
print('torch', torch.__version__, 'cuda', ok, name, 'capability', cap2, 'arch', arch, flush=True)

In [ ]:
# ── 实验队列(独立可重排;想换实验只改这里)──
PARITY = ('python benchmarks/reasoning_depth.py --task parity --difficulty 32 '
          '--mix --eval_depths 1 --attention_layers ')
QUEUE = [
    # A. parity k=32 selective 长预算(纯 LNN,12k 步 — 本地 6000 步全 chance=预算墙)
    PARITY + '--seeds 0 1 2 --steps 12000 --selective_decay --skip_transformer --tag kg-parity-d32-sel',
    # B. parity k=32 stock 同预算对照(预言: 仍卡 chance,确认预算墙非机制)
    PARITY + '--seeds 0 1 2 --steps 12000 --tag kg-parity-d32-stock',
    # C. J-Space J1 工作区驻留 sweep(pointer_chase 课程任务,workspace_iterations 深度)
    'python benchmarks/reasoning_depth.py --task pointer_chase --n_values 8 --mode fixed '
    '--mix --difficulty 4 --steps 10000 --seeds 0 1 2 --eval_depths 1 2 4 --workspace --tag kg-j1',
    # D. 125M selective_decay 文本实验(文本翻盘关键证据)
    #    P100 + cu118 torch 可跑;CPU 兜底会跑数周 — 能力不足时跳过并留档。
    #    注意: scaling_comparison.py 无 --tag,结果落 out_dir 的 JSON + run.log。
    'python benchmarks/scaling_comparison.py --mode train --archs mt_lnn --steps 20000 '
    '--seeds 0,1,2 --dtype fp32 --ckpt_every 500 --resume --selective_decay '
    '--train_token_cap 50000000 --out_dir scaling_fp32/p0_2b_selective',
]
START = 0  # 断点续跑:改成上次完成的序号+1

# 125M 实验需要 GPU(sm_60+ 即可,P100+cu118 torch 已解锁);纯 CPU 时跳过
# ⚠ 用子进程探测 cuda: pip 装新 torch 后当前进程的 import 仍是旧版,
#   队列本身走 subprocess(新 torch),所以 GPU_OK 也要以子进程为准。
GPU_OK = False
try:
    probe = subprocess.run([sys.executable, '-c',
        'import torch; a=torch.cuda.is_available(); '
        'print(a, torch.cuda.get_device_capability(0) if a else (0,0))'],
        capture_output=True, text=True, timeout=120)
    parts = probe.stdout.strip().split()
    if len(parts) >= 2 and parts[0] == 'True':
        caps = parts[1].strip('()').split(',')
        GPU_OK = int(caps[0]) >= 6
    print('probe cuda:', probe.stdout.strip(), flush=True)
except Exception as e:
    print('probe err:', e, flush=True)
if not GPU_OK:
    print(f'⚠ GPU 不可用 — 125M 实验需要 GPU,跳过队列 D', flush=True)
    QUEUE = QUEUE[:3]

def snapshot():
    for f in glob.glob('benchmarks/results/*.jsonl') + glob.glob('benchmarks/results/*.log'):
        shutil.copy(f, '/kaggle/working/')
    # 125M 实验的 out_dir 结果也拷出
    for f in glob.glob('scaling_fp32/**/*.json', recursive=True) + glob.glob('scaling_fp32/**/run.log', recursive=True):
        try:
            rel = os.path.relpath(f, 'scaling_fp32')
            dst = os.path.join('/kaggle/working/', rel.replace(os.sep, '__'))
            shutil.copy(f, dst)
        except Exception:
            pass

for i, cmd in enumerate(QUEUE):
    if i < START: continue
    print(f'\n══════ [{i}] {cmd}\n', flush=True)
    t0 = time.time()
    r = subprocess.run(cmd.split(), capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0: print('STDERR:', r.stderr[-2000:])
    print(f'[{i}] 用时 {(time.time()-t0)/60:.1f} min, exit={r.returncode}', flush=True)
    snapshot()
print('\n全部完成')

In [ ]:
# ── 打包下载 ──
import shutil
shutil.make_archive('/kaggle/working/m1_results', 'zip', '/kaggle/working/M1/benchmarks/results')
print('下载 /kaggle/working/m1_results.zip,发回给 CC 入库')